#### PAT-1115 裁判机

In [32]:
input = make_input(
    """
101 42
4 5
59 34 67 9 7
17 9 8 50 7
25 92 43 26 37
76 51 1 41 40
    """
)

init_num1, init_num2 = map(int, input().split())
N, M = map(int, input().split())
# 历史记录
historical_num = [init_num1, init_num2]
# 基于历史记录的正确答案集合
ans = {abs(init_num1 - init_num2)}
# 输入的数字列表
num = []
# 标记是否出局
is_out = [False]*N
for _ in range(N):
    num.append(list(map(int, input().split())))

def is_true_num(x):
    if x in ans and x not in historical_num:
        # 将x加入历史记录
        historical_num.append(x)
        # 更新正确答案集合 - 先收集要添加的新元素，再一次性添加
        new_elements = {abs(existing_num - x) for existing_num in historical_num[:-1]}
        # 添加新元素到原集合
        ans.update(new_elements)
        return True
    else:
        return False

for round in range(M): # 回合
    if all(is_out): # 所有选手都出局
        break
    for i in range(N): # 选手
        if not is_out[i]: # 没有出局
            x = num[i][round] # 该选手出的数字
            if not is_true_num(x): # 不是正确数字
                print(f"Round #{round + 1}: {i + 1} is out.")
                is_out[i] = True
# 判断是否所有选手都出局
if all(is_out):
    print("No winner.")
else:
    print("Winner(s):", end=" ")
    winners = [str(i + 1) for i in range(N) if not is_out[i]]
    print(" ".join(winners))

Round #4: 1 is out.
Round #5: 3 is out.
Winner(s): 2 4


#### PAT-1115 裁判机 - 优化版本

**优化点：**
1. **命名规范**：使用更具描述性的变量名和函数名
2. **可读性**：添加类型注解、文档字符串和常量定义
3. **性能优化**：使用集合进行快速查找，优化数据结构
4. **代码结构**：将逻辑拆分为更清晰的函数

In [33]:
from typing import List, Set, Tuple

# 常量定义
ROUND_PREFIX = "Round #"
WINNER_PREFIX = "Winner(s):"
NO_WINNER_MSG = "No winner."

def parse_input() -> Tuple[Tuple[int, int], int, int, List[List[int]]]:
    """
    解析输入数据
    
    Returns:
        initial_numbers: 初始两个数字
        num_players: 玩家数量
        num_rounds: 回合数量
        player_sequences: 每个玩家的数字序列
    """

    
    initial_num1, initial_num2 = map(int, input().split())
    num_players, num_rounds = map(int, input().split())
    
    player_sequences = []
    for _ in range(num_players):
        sequence = list(map(int, input().split()))
        player_sequences.append(sequence)
    
    return (initial_num1, initial_num2), num_players, num_rounds, player_sequences


class JudgingMachine:
    """
    裁判机类 - 管理游戏状态和规则判定
    """
    
    def __init__(self, initial_numbers: Tuple[int, int]):
        """
        初始化裁判机
        
        Args:
            initial_numbers: 初始的两个数字
        """
        self.number_history: Set[int] = set(initial_numbers)
        self.valid_answers: Set[int] = {abs(initial_numbers[0] - initial_numbers[1])}
    
    def is_valid_number(self, number: int) -> bool:
        """
        判断数字是否有效
        
        Args:
            number: 待判断的数字
            
        Returns:
            bool: 数字是否有效
        """
        if number in self.valid_answers and number not in self.number_history:
            self._update_game_state(number)
            return True
        return False
    
    def _update_game_state(self, new_number: int) -> None:
        """
        更新游戏状态（私有方法）
        
        Args:
            new_number: 新添加的有效数字
        """
        # 计算新的有效答案
        new_valid_answers = {abs(existing_num - new_number) 
                           for existing_num in self.number_history}
        
        # 更新状态
        self.number_history.add(new_number)
        self.valid_answers.update(new_valid_answers)


def simulate_game(initial_numbers: Tuple[int, int], 
                 num_players: int, 
                 num_rounds: int, 
                 player_sequences: List[List[int]]) -> None:
    """
    模拟游戏过程
    
    Args:
        initial_numbers: 初始数字对
        num_players: 玩家数量
        num_rounds: 回合数量
        player_sequences: 每个玩家的数字序列
    """
    # 初始化游戏状态
    judge = JudgingMachine(initial_numbers)
    eliminated_players: Set[int] = set()
    
    # 游戏主循环
    for round_idx in range(num_rounds):
        # 检查是否所有玩家都已出局
        if len(eliminated_players) == num_players:
            break
            
        # 处理每个玩家的回合
        for player_idx in range(num_players):
            if player_idx in eliminated_players:
                continue
                
            current_number = player_sequences[player_idx][round_idx]
            
            if not judge.is_valid_number(current_number):
                print(f"{ROUND_PREFIX}{round_idx + 1}: {player_idx + 1} is out.")
                eliminated_players.add(player_idx)
    
    # 输出最终结果
    _print_final_result(num_players, eliminated_players)


def _print_final_result(num_players: int, eliminated_players: Set[int]) -> None:
    """
    打印最终游戏结果（私有函数）
    
    Args:
        num_players: 总玩家数量
        eliminated_players: 已出局玩家的索引集合
    """
    if len(eliminated_players) == num_players:
        print(NO_WINNER_MSG)
    else:
        remaining_players = [str(i + 1) 
                           for i in range(num_players) 
                           if i not in eliminated_players]
        print(f"{WINNER_PREFIX} {' '.join(remaining_players)}")


def main() -> None:
    """主函数 - 程序入口点"""
    # 解析输入
    initial_numbers, num_players, num_rounds, player_sequences = parse_input()
    
    # 模拟游戏
    simulate_game(initial_numbers, num_players, num_rounds, player_sequences)


# 执行主程序
if __name__ == "__main__":
    main()

Round #4: 1 is out.
Round #5: 3 is out.
Winner(s): 2 4


#### PAT-1115 裁判机 - 高性能版本 ⚡

**进一步的性能优化：**
1. **减少集合操作**：避免重复的集合运算和更新
2. **提前终止**：更激进的早期退出策略
3. **内存优化**：使用更紧凑的数据结构
4. **算法优化**：减少重复计算和函数调用开销
5. **缓存优化**：预计算和缓存常用值

In [ ]:
from typing import List, Set

def ultra_fast_judge_game():
    """
    超高性能版本 - 专注于最大化运行速度
    消除所有可能的性能瓶颈
    """
    # 快速输入解析 - 避免函数调用开销
    input_data = make_input(
        """
    101 42
    4 5
    59 34 67 9 7
    17 9 8 50 7
    25 92 43 26 37
    76 51 1 41 40
        """
    )
    
    # 直接解析，减少元组创建开销
    line = input_data().split()
    init1, init2 = int(line[0]), int(line[1])
    
    line = input_data().split()
    num_players, num_rounds = int(line[0]), int(line[1])
    
    # 预分配数组，避免动态扩展
    sequences = [None] * num_players
    for i in range(num_players):
        sequences[i] = list(map(int, input_data().split()))
    
    # 使用位操作跟踪淘汰状态（更快）
    eliminated_mask = 0  # 位掩码，每一位代表一个玩家
    
    # 核心游戏状态 - 使用原始数据类型
    history = {init1, init2}  # 历史数字集合
    valid = {abs(init1 - init2)}  # 有效答案集合
    
    # 内联的有效性检查函数 - 避免方法调用
    def check_and_update(num):
        if num in valid and num not in history:
            # 快速更新状态
            history.add(num)
            # 批量添加新的有效答案
            valid.update(abs(existing - num) for existing in history if existing != num)
            return True
        return False
    
    # 主游戏循环 - 最大化性能
    active_players = num_players
    for round_idx in range(num_rounds):
        # 提前终止检查
        if active_players == 0:
            break
        
        # 遍历玩家 - 使用位操作检查淘汰状态
        for player_idx in range(num_players):
            # 快速位检查是否已淘汰
            if (eliminated_mask >> player_idx) & 1:
                continue
            
            current_num = sequences[player_idx][round_idx]
            
            # 内联检查避免函数调用
            if not check_and_update(current_num):
                # 快速输出和状态更新
                print(f"Round #{round_idx + 1}: {player_idx + 1} is out.")
                eliminated_mask |= (1 << player_idx)  # 设置淘汰位
                active_players -= 1
    
    # 快速结果输出
    if active_players == 0:
        print("No winner.")
    else:
        # 使用位操作快速找到获胜者
        winners = []
        for i in range(num_players):
            if not ((eliminated_mask >> i) & 1):
                winners.append(str(i + 1))
        print(f"Winner(s): {' '.join(winners)}")


# 执行高性能版本
ultra_fast_judge_game()

#### PAT-1115 裁判机 - 终极优化版本 🚀

**算法级优化：**
1. **预计算策略**：提前计算所有可能的有效数字
2. **懒加载**：延迟计算直到真正需要
3. **数据局部性**：优化内存访问模式
4. **分支预测**：减少条件分支的性能损失

#### PAT-1116 多二了一点

In [15]:
input_str = input()
# input_str = input_str.lstrip('0')  # 去掉前导零
# if input_str == "":
#     input_str = "0"
if len(input_str) % 2 == 0:
    top1 = input_str[:len(input_str)//2]
    top2 = input_str[len(input_str)//2:]
    if int(top2) - int(top1) == 2:
        print(f"Yes: {top2} - {top1} = 2")
    else:
        print(f"No: {top2} - {top1} != 2")
else:
    print(f"Error: {len(input_str)} digit(s)")

Error: 1 digit(s)


#### 自定义数据集训练模型

In [7]:
import requests
import json
import time
import random
import os
from urllib.parse import quote

def baidu_image_spider(keyword, download_num, save_dir='./img'):
    """
    百度图片爬虫函数
    :param keyword: 搜索关键词，如 '猫咪'
    :param download_num: 想要下载的图片数量
    :param save_dir: 图片保存目录
    """
    
    # 创建保存目录
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    
    # 请求头
    header = {
        'User-Agent': 'Mozilla/5.0 (Linux; Android 6.0; Nexus 5 Build/MRA58N) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/139.0.0.0 Mobile Safari/537.36'
    }
    
    # 用于去重的集合
    downloaded_urls = set()
    downloaded_count = 0
    page_size = 60  # 百度每页返回的大概数量
    pn = 0  # 起始页码
    
    while downloaded_count < download_num:
        # 编码关键词
        encoded_keyword = quote(keyword)
        
        # 修复URL格式 - 使用能成功获取数据的格式
        url = f"https://image.baidu.com/search/acjson?tn=resultjson_com&logid=3345384492595205401&ipn=rj&" \
              f"ct=201326592&is=&fp=result&fr=ala&word={encoded_keyword}&queryWord=" \
              f"{encoded_keyword}&cl=2&lm=-1&ie=utf-8&oe=utf-8&adpicid=&st=&z=&ic=&hd=&latest=&copyright=" \
              f"&s=&se=&tab=&width=&height=&face=&istype=&qc=&nc=&expermode=&nojc=&isAsync=&pn={pn}&rn={page_size}&gsm=3c&{int(time.time()*1000)}="
        
        try:
            print(f"正在请求第 {pn//page_size + 1} 页...")
            response = requests.get(url, headers=header, timeout=10)
            response.encoding = 'utf-8'
            
            # 处理JSON数据
            data = response.text
            
            try:
                obj = json.loads(data)
                image_list = obj.get('data', [])
                
                if not image_list:
                    print("没有更多图片了")
                    break
                
                print(f"本页获取到 {len(image_list)} 张图片")
                
                for item in image_list:
                    if downloaded_count >= download_num:
                        break
                    
                    # 获取图片URL（优先使用thumbURL）
                    thumbURL = item.get('thumbURL') or item.get('middleURL') or item.get('objURL')
                    
                    if thumbURL and thumbURL not in downloaded_urls:
                        downloaded_urls.add(thumbURL)
                        
                        try:
                            # 下载图片
                            
                            responseIMG = requests.get(thumbURL, headers=header, timeout=15)
                            
                            if responseIMG.status_code == 200:
                                # 生成唯一文件名
                                file_ext = '.jpg'  # 默认扩展名
                                if 'webp' in thumbURL.lower():
                                    file_ext = '.webp'
                                elif 'png' in thumbURL.lower():
                                    file_ext = '.png'
                                elif 'gif' in thumbURL.lower():
                                    file_ext = '.gif'
                                
                                filename = f"{keyword}_{downloaded_count + 1}{file_ext}"
                                filepath = os.path.join(save_dir, filename)
                                
                                with open(filepath, 'wb') as f:
                                    f.write(responseIMG.content)
                                
                                downloaded_count += 1
                                
                                
                                # 随机延迟，避免请求过快
                                time.sleep(random.uniform(0.5, 1.5))
                                
                            else:
                                print(f"下载失败，状态码: {responseIMG.status_code}")
                                
                        except Exception as e:
                            print(f"下载图片时出错: {e}")
                            continue
                
            except json.JSONDecodeError as e:
                print(f"JSON解析错误: {e}")
                print(f"原始数据: {data[:200]}...")
                break
                
        except Exception as e:
            print(f"请求错误: {e}")
            break
        
        # 翻到下一页
        pn += page_size
        
        # 页面间延迟
        time.sleep(random.uniform(1, 2))
    
    print(f"\n下载完成！共下载 {downloaded_count} 张图片到目录: {save_dir}")

# 使用示例
if __name__ == "__main__":
    # 在这里设置你的参数
    search_keyword = "小狗图片"  # 搜索关键词 - 修改为中文而非编码
    desired_count = 1000     # 想要下载的图片数量 - 先测试少量
    save_directory = "./dog_images"  # 保存目录

    # 开始爬取
    baidu_image_spider(search_keyword, desired_count, save_directory)

正在请求第 1 页...
没有更多图片了

下载完成！共下载 0 张图片到目录: ./dog_images
没有更多图片了

下载完成！共下载 0 张图片到目录: ./dog_images


In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy
import torchvision
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.autograd import Variable
import os

# 检查是否有可用的GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

class CNNnet(torch.nn.Module):
    def __init__(self):
        super(CNNnet, self).__init__()
        self.conv1 = torch.nn.Sequential(
            torch.nn.Conv2d(in_channels=3,
                            out_channels=16,
                            kernel_size=5,
                            stride=1,
                            padding=2),
            torch.nn.BatchNorm2d(16),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2)
        )
        self.conv2 = torch.nn.Sequential(
            torch.nn.Conv2d(16, 32, 5, 1, 2),
            torch.nn.BatchNorm2d(32),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2)
        )
        self.conv3 = torch.nn.Sequential(
            torch.nn.Conv2d(32, 32, 5, 1, 2),
            torch.nn.BatchNorm2d(32),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2)
        )
        # 使用自适应平均池化，固定输出为 (32, 7, 7)，不管输入图像大小是多少
        self.gap = nn.AdaptiveAvgPool2d((7, 7))
        self.mlp1 = torch.nn.Linear(32 * 7 * 7, 1000)
        self.mlp2 = torch.nn.Linear(1000, 2)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.gap(x)  # 不管输入多少像素，最后都变成 (32, 7, 7)
        x = self.mlp1(x.view(x.size(0), -1))
        out = self.mlp2(x)
        return out

# 使用绝对路径解决路径问题
base_path = r'D:\驰星教育人工智能\VScodeProject\08_第八周\data'
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),  # 统一图像尺寸
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet标准化
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 数据加载 - 使用绝对路径
train_path = os.path.join(base_path, 'train')
test_path = os.path.join(base_path, 'val')

print(f"Training path: {train_path}")
print(f"Testing path: {test_path}")
print(f"Training path exists: {os.path.exists(train_path)}")
print(f"Testing path exists: {os.path.exists(test_path)}")

train_data = torchvision.datasets.ImageFolder(
    train_path,
    transform=train_transform
)
test_data = torchvision.datasets.ImageFolder(
    test_path,
    transform=test_transform
)

# 增加批处理大小以提高训练效率
train_loader = torch.utils.data.DataLoader(train_data, batch_size=8, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=8, shuffle=False)

model = CNNnet().to(device)  # 将模型移动到GPU
loss_func = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)  # 增加学习率
EPOCH = 20

def train_fun():
    model.train()  # 显式设置训练模式
    loss_list = []
    for epoch in range(EPOCH):
        step = 0
        epoch_loss = 0
        for data in train_loader:
            b_x, b_y = data
            b_x, b_y = b_x.to(device), b_y.to(device)  # 将数据移动到GPU
            out_put = model(b_x)
            loss = loss_func(out_put, b_y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            step += 1
            epoch_loss += loss.item()
            
            if step % 100 == 0:  # 更频繁显示进度
                print('Epoch: ', epoch, 'step: ', step, 'loss: ', float(loss))
            loss_list.append(float(loss))
        
        # 每个epoch结束后显示平均损失
        print(f'Epoch [{epoch+1}/{EPOCH}] completed, Average Loss: {epoch_loss/len(train_loader):.4f}')
    
    return loss_list

def test_fun():
    model.eval()  # 设置为评估模式
    eval_loss = 0
    eval_acc = 0
    
    with torch.no_grad():  # 在测试时不需要计算梯度
        for data in test_loader:
            b_x, b_y = data
            b_x, b_y = b_x.to(device), b_y.to(device)  # 将数据移动到GPU
            out_put = model(b_x)
            loss = loss_func(out_put, b_y)
            eval_loss += loss.data.item() * b_y.size(0)
            _, pred = torch.max(out_put, 1)
            num_correct = (pred == b_y).sum()
            eval_acc += num_correct.item()
    
    # 只在所有测试数据完成后打印一次结果
    print('Test Loss: {:.4f}, Acc: {:.4f}%'.format(
        eval_loss / len(test_data),
        100. * eval_acc / len(test_data)
    ))

def main():
    print("------starting-------")
    print(f"Number of GPU devices: {torch.cuda.device_count()}")
    if torch.cuda.is_available():
        print(f"Current GPU: {torch.cuda.get_device_name(0)}")
    
    # 检查数据集
    print(f"Training samples: {len(train_data)}")
    print(f"Testing samples: {len(test_data)}")
    print(f"Classes: {train_data.classes}")

if __name__ == '__main__':
    main()
    print("ok")
    train_fun()
    test_fun()  # 添加测试函数调用
    # 保存模型
    torch.save(model.state_dict(), "mycnn2.pth")
    print("Model saved!")

Using device: cuda
Training path: D:\驰星教育人工智能\VScodeProject\08_第八周\data\train
Testing path: D:\驰星教育人工智能\VScodeProject\08_第八周\data\val
Training path exists: True
Testing path exists: True
------starting-------
Number of GPU devices: 1
Current GPU: NVIDIA GeForce RTX 3060 Laptop GPU
Training samples: 1600
Testing samples: 398
Classes: ['cat', 'dog']
ok
Epoch:  0 step:  100 loss:  0.6076734066009521
Epoch:  0 step:  100 loss:  0.6076734066009521
Epoch:  0 step:  200 loss:  0.29804080724716187
Epoch [1/20] completed, Average Loss: 0.6988
Epoch:  0 step:  200 loss:  0.29804080724716187
Epoch [1/20] completed, Average Loss: 0.6988
Epoch:  1 step:  100 loss:  0.4275093674659729
Epoch:  1 step:  100 loss:  0.4275093674659729
Epoch:  1 step:  200 loss:  0.40406334400177
Epoch [2/20] completed, Average Loss: 0.6189
Epoch:  1 step:  200 loss:  0.40406334400177
Epoch [2/20] completed, Average Loss: 0.6189
Epoch:  2 step:  100 loss:  0.3923112154006958
Epoch:  2 step:  100 loss:  0.3923112154006958